# Queue throughput by buffer size

Throughput is calculated from the benchmark timestamps as `2^24 / (EndTimestamp - StartTimestamp)`. Each point is the mean across the available runs for that queue variant and buffer size.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd

import os # For getting required environment variables

In [ ]:
# Try getting the environment variables.

try:
    # Crashes if the environment variable is not set.
    NUM_RUNS = int(os.environ["NUM_RUNS"])
    OUTPUT_DIR = os.environ["OUTPUT_DIR"]
    SAMPLE_INTERVAL = int(os.environ["SAMPLE_INTERVAL"])
    NUM_ELEMENTS = int(os.environ["NUM_ELEMENTS"])

except KeyError as e:
    print(f"Environment variable {e} not found. Please set it before running the script.")
    exit(1)

In [ ]:
base_filename = "latency_throughput_buffer_"

result_root = Path(OUTPUT_DIR)
csv_files = sorted(result_root.glob("*/run-*/" + base_filename + "*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No benchmark CSV files found under {result_root.resolve()}")

In [ ]:
metadata_pattern = re.compile(
    r"StartTimestamp=(?P<start>[0-9.]+)\s+"
    r"EndTimestamp=(?P<end>[0-9.]+)"
)
records = []

# Directory structure: "results/<variant>/run-<run_number>/latency_throughput_buffer_<buffer_size>.csv"

for file in csv_files:
    with file.open() as handle:
        metadata = handle.readline().strip() # Read the first line of the CSV file, which contains the start and end timestamps.

    match = metadata_pattern.search(metadata) # Search for the start and end timestamps in the metadata line using the regex pattern.
    if match is None:
        raise ValueError(f"Missing benchmark timestamps in {file}")

    start = float(match.group("start")) # Convert the start timestamp to a float.
    end = float(match.group("end")) # Convert the end timestamp to a float.
    elapsed = end - start # Calculate the elapsed time of the benchmark by subtracting the start timestamp from the end timestamp.
    if elapsed <= 0: # Check if the elapsed time is non-positive, which would indicate an error in the benchmark timing.
        raise ValueError(f"Non-positive benchmark duration in {file}: {elapsed}")

    variant = file.parts[-3] # Extract the variant name from the file path. The variant is the third-to-last part of the path (e.g., "variant/run-1/latency_throughput_buffer_1024.csv").
    ordering = "seq-cst" if variant.endswith("-seq_cst") else "acq-rel" # Determine the ordering type based on the variant name. If the variant ends with "-seq_cst", it is sequentially consistent; otherwise, it is acquire-release.
    family = variant.removesuffix("-seq_cst") # Remove the "-seq_cst" suffix from the variant name to get the family name (e.g., "variant-seq_cst" becomes "variant").
    buffer_size = int(file.stem.removeprefix(base_filename)) # Extract the buffer size from the file name. The buffer size is the part of the file name after "latency_throughput_buffer_" (e.g., "latency_throughput_buffer_1024.csv" becomes 1024).
    run = file.parts[-2] # Extract the run number from the file path. The run number is the second-to-last part of the path (e.g., "variant/run-1/latency_throughput_buffer_1024.csv" becomes "run-1").
    
    records.append(
        {
            "variant": variant,
            "family": family,
            "ordering": ordering,
            "buffer_size": buffer_size,
            "run": run,
            "throughput": NUM_ELEMENTS / elapsed,
        }
    )

# Create a table with one row per buffer size and one column per variant.
throughput_df = pd.DataFrame(records)

throughput_df.pivot_table( # Pivot the DataFrame to create a table with buffer sizes as rows and variants as columns, where the values are the mean throughput for each combination of buffer size and variant.
    index="buffer_size",
    columns="variant",
    values="throughput",
    aggfunc="mean",
).sort_index()

throughput_df = pd.DataFrame(records)

In [ ]:
summary = (
    throughput_df.groupby(["family", "ordering", "buffer_size"], as_index=False)
    .agg(
        throughput=("throughput", "mean"),
        runs=("throughput", "size"),
    )
    .sort_values(["family", "ordering", "buffer_size"])
)

families = sorted(summary["family"].unique())
colors = dict(zip(families, plt.get_cmap("tab10").colors))
line_styles = {"acq-rel": "-", "seq-cst": ":"}

fig, ax = plt.subplots(figsize=(12, 7))
for (family, ordering), group in summary.groupby(["family", "ordering"]):
    group = group.sort_values("buffer_size")
    ax.plot(
        group["buffer_size"],
        group["throughput"],
        color=colors[family],
        linestyle=line_styles[ordering],
        marker="o",
        label=f"{family} ({ordering})",
    )

ax.set_xscale("log", base=2)
ax.set_xlabel("Ring buffer size (number of queue slots)")
ax.set_ylabel("Throughput (queue operations / second)")
ax.set_title("Lockfree SPSC queue throughput by buffer size \n (averaged over " + str(NUM_RUNS) + " runs, " + str(NUM_ELEMENTS) + " push-pop operations per run)")
ax.grid(True, which="both", alpha=0.25)
ax.legend(title="Queue variant")
fig.tight_layout()
plt.savefig(OUTPUT_DIR + "/output.png", dpi=300, bbox_inches="tight")